# 03 — Exposure Labels v2

Identifies anchor posts and classifies users as exposed or unexposed using correct thread-level linking via `link_id`.

**Anchor post definition:**
- Post is in the anchor period: Sep 1–Nov 30, 2023 (cycle 1) or Sep 1–Nov 30, 2024 (cycle 2)
- Post matches negative keyword list (rejection language, re-applicant discourse, anxiety/stress/depression terms)
- Post scores above the **upper tertile (p67)** on **at least 1 of 3** SVM classifier dimensions (anxiety, depression, stress)
- Thresholds are computed empirically from the anchor-period keyword-filtered posts, not fixed manually

**Anchor post intensity (`n_dims`)**: number of dimensions (1–3) on which the post exceeds its tertile threshold.
Used as a continuous dose variable in downstream analysis.

**Exposed user**: commented on an anchor post thread (identified via `link_id`); anchor post authors excluded.
Each exposed user is assigned `exposure_intensity` = max `n_dims` across the anchor threads they engaged with.

**Unexposed user**: active in r/gradadmissions during Aug 1–May 31 of that cycle but never commented on an anchor thread

**Inputs:**
- `data/processed_v2/posts_clean.jsonl` + `comments_clean.jsonl` (from notebook 01)
- `models/clf_anxiety.joblib`, `clf_depression.joblib`, `clf_stress.joblib` (from notebook 02)

**Outputs:**
- `data/processed_v2/anchor_posts_v2.parquet` — includes `n_dims` column
- `data/processed_v2/exposure_labels_v2.parquet` — `author, exposed (bool), cycle, exposure_intensity`


In [1]:
import json
import pandas as pd
import numpy as np
import joblib
import re
from pathlib import Path

ROOT         = Path('..').resolve()
DATA_DIR     = ROOT / 'data' / 'processed_v2'
MODEL_DIR    = ROOT / 'models'

# Source files (from NB01 cleaned output and raw comments)
POSTS_PATH    = ROOT / 'cleaned_output' / 'r_gradadmissions_posts.cleaned.jsonl'
COMMENTS_PATH = ROOT / 'Grad Admissions Comments.jsonl'

CYCLES = {
    1: {
        'anchor_start': '2023-09-01',
        'anchor_end':   '2023-11-30',
        'active_start': '2023-08-01',
        'active_end':   '2024-05-31',
    },
    2: {
        'anchor_start': '2024-09-01',
        'anchor_end':   '2024-11-30',
        'active_start': '2024-08-01',
        'active_end':   '2025-05-31',
    },
}

def load_jsonl(path):
    rows = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

print('Paths:')
print(' Posts:   ', POSTS_PATH)
print(' Comments:', COMMENTS_PATH)
print(' Models:  ', MODEL_DIR)
print(' Output:  ', DATA_DIR)


Paths:
 Posts:    /Users/veda/Documents/reddit-gradadmissions-distress/cleaned_output/r_gradadmissions_posts.cleaned.jsonl
 Comments: /Users/veda/Documents/reddit-gradadmissions-distress/Grad Admissions Comments.jsonl
 Models:   /Users/veda/Documents/reddit-gradadmissions-distress/models
 Output:   /Users/veda/Documents/reddit-gradadmissions-distress/data/processed_v2


## 1) Load raw posts

In [2]:
raw_posts = load_jsonl(POSTS_PATH)
posts = pd.DataFrame([{
    'id':           r['record_id'],
    'author':       r['author'],
    'created_dt':   pd.Timestamp(r['created_utc'], unit='s', tz='UTC'),
    'clean_text':   r.get('clean_text', ''),
    'score':        r.get('score', 0),
    'num_comments': r.get('num_comments', 0),
} for r in raw_posts])
print(f'Clean posts loaded: {len(posts):,} from {posts["author"].nunique():,} unique authors')
print(f'Date range: {posts["created_dt"].min().date()} → {posts["created_dt"].max().date()}')


Clean posts loaded: 88,441 from 45,309 unique authors
Date range: 2023-08-01 → 2025-07-30


## 2) Filter to anchor periods and score with SVM classifiers

In [3]:
# Tag each post with its cycle (if in an anchor period)
def assign_cycle(dt):
    for cycle, w in CYCLES.items():
        if pd.Timestamp(w['anchor_start'], tz='UTC') <= dt <= pd.Timestamp(w['anchor_end'] + ' 23:59:59', tz='UTC'):
            return cycle
    return None

posts['cycle'] = posts['created_dt'].apply(assign_cycle)
anchor_candidates = posts[posts['cycle'].notna()].copy()

print(f'Posts in anchor periods: {len(anchor_candidates):,}')
print(anchor_candidates['cycle'].value_counts().sort_index())

Posts in anchor periods: 15,973
cycle
1.0    7786
2.0    8187
Name: count, dtype: int64


In [4]:
# Load SVM classifiers
clf_anx = joblib.load(MODEL_DIR / 'clf_anxiety.joblib')
clf_dep = joblib.load(MODEL_DIR / 'clf_depression.joblib')
clf_str = joblib.load(MODEL_DIR / 'clf_stress.joblib')
print('Classifiers loaded.')

texts = anchor_candidates['clean_text'].tolist()

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

anchor_candidates['anx_score'] = sigmoid(clf_anx.decision_function(texts))
anchor_candidates['dep_score'] = sigmoid(clf_dep.decision_function(texts))
anchor_candidates['str_score'] = sigmoid(clf_str.decision_function(texts))
anchor_candidates['mean_mh_score'] = anchor_candidates[['anx_score', 'dep_score', 'str_score']].mean(axis=1)

print(f'Scored {len(anchor_candidates):,} anchor-period posts')
print(anchor_candidates['mean_mh_score'].describe().round(4))


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/base.py:440: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.8.0 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/base.py:440: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.8.0 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/base.py:440: InconsistentVersionWarning: Tr

Classifiers loaded.


Scored 15,973 anchor-period posts
count    15973.0000
mean         0.3596
std          0.0849
min          0.0843
25%          0.3009
50%          0.3545
75%          0.4124
max          0.7477
Name: mean_mh_score, dtype: float64


## 3) Apply keyword filter → anchor posts

In [5]:
NEGATIVE_KEYWORDS = [
    r'\breject(?:ed|ion)\b',
    r'\bdeclin(?:ed|ing)\b',
    r'\bwaitlist(?:ed)?\b',
    r'\bfunding\s+(?:lost|cut|removed|denied|gap|issue)\b',
    r'\bno\s+funding\b',
    r'\bstipend\b',
    r'\bwithdrew?\s+(?:offer|admission)\b',
    r'\bacceptance\s+rate\b',
    r'\bno\s+(?:offer|response|interview)\b',
    r'\bsilence\s+from\b',
    r'\bnot\s+(?:accepted|admitted|selected)\b',
    r'\bgave\s+up\b',
    r'\bmental\s+health\b',
    r'\banxi(?:ous|ety)\b',
    r'\bdepress(?:ed|ing|ion)\b',
    r'\bstress(?:ed|ful)?\b',
    r'\boverwhelm(?:ed|ing)\b',
    r'\bscared\b',
    r'\bworr(?:ied|ying)\b',
    r'\bfalling\s+apart\b',
    r'\bbreaking\s+down\b',
    r"\bcan(?:'t|not)\s+(?:take|handle|cope)\b",
    r'\bno\s+chance\b',
    r'\bnot\s+good\s+enough\b',
    r'\bregret\b',
    r'\bfailed\b',
    r'\bimposter\b',
]

keyword_pattern = re.compile('|'.join(NEGATIVE_KEYWORDS), re.IGNORECASE)

anchor_candidates['has_neg_keyword'] = anchor_candidates['clean_text'].str.contains(
    keyword_pattern, na=False
)
kw_filtered = anchor_candidates[anchor_candidates['has_neg_keyword']].copy()

# Compute per-dimension upper tertile (p67) thresholds empirically
# from the keyword-filtered anchor-period posts themselves
DIMS = ['anx_score', 'dep_score', 'str_score']
tertile_thresholds = {dim: kw_filtered[dim].quantile(2/3) for dim in DIMS}
print('Per-dimension upper tertile thresholds (p67):')
for dim, thr in tertile_thresholds.items():
    print(f'  {dim}: {thr:.4f}')

# Flag each dimension and count how many fire per post
for dim in DIMS:
    kw_filtered[f'{dim}_above'] = kw_filtered[dim] > tertile_thresholds[dim]
kw_filtered['n_dims'] = (
    kw_filtered['anx_score_above'].astype(int) +
    kw_filtered['dep_score_above'].astype(int) +
    kw_filtered['str_score_above'].astype(int)
)

# Anchor posts: keyword filter + at least 1 of 3 dimensions above tertile
anchor_posts = kw_filtered[kw_filtered['n_dims'] >= 1].copy()

print(f'\nAnchor posts identified: {len(anchor_posts):,}')
print(anchor_posts['cycle'].value_counts().sort_index())
print(f'Unique anchor authors: {anchor_posts["author"].nunique():,}')
print(f'\nn_dims distribution (distress intensity):')
print(anchor_posts['n_dims'].value_counts().sort_index().rename('anchor_posts'))


Per-dimension upper tertile thresholds (p67):
  anx_score: 0.4384
  dep_score: 0.4066
  str_score: 0.4703

Anchor posts identified: 1,024
cycle
1.0    458
2.0    566
Name: count, dtype: int64
Unique anchor authors: 934

n_dims distribution (distress intensity):
n_dims
1    309
2    216
3    499
Name: anchor_posts, dtype: int64


In [6]:
# Save anchor posts (includes n_dims intensity column)
anchor_posts[[
    'id', 'author', 'created_dt', 'cycle', 'clean_text',
    'anx_score', 'dep_score', 'str_score', 'mean_mh_score',
    'n_dims', 'score', 'num_comments'
]].to_parquet(DATA_DIR / 'anchor_posts_v2.parquet', index=False)
print('Saved anchor_posts_v2.parquet')

# Anchor post ID sets per cycle
anchor_ids_by_cycle = {
    cycle: set(anchor_posts[anchor_posts['cycle'] == cycle]['id'])
    for cycle in [1, 2]
}
print(f'Anchor IDs — cycle 1: {len(anchor_ids_by_cycle[1]):,}, cycle 2: {len(anchor_ids_by_cycle[2]):,}')

# Build n_dims lookup: post_id -> n_dims (for assigning exposure intensity to users)
post_ndims = anchor_posts.set_index('id')['n_dims'].to_dict()

# Anchor authors per cycle (to exclude from exposed set)
anchor_authors_by_cycle = {
    cycle: set(anchor_posts[anchor_posts['cycle'] == cycle]['author'])
    for cycle in [1, 2]
}


Saved anchor_posts_v2.parquet
Anchor IDs — cycle 1: 458, cycle 2: 566


## 4) Load comments → identify exposed users via `link_id`

In [7]:
import datetime

# Load raw comments and extract link_id (strip 't3_' prefix) as post_id
print('Loading comments (this may take ~30s)...')
comment_rows = []
with open(COMMENTS_PATH) as f:
    for line in f:
        r = json.loads(line)
        comment_rows.append({
            'id':         r.get('id', ''),
            'author':     r.get('author', ''),
            'post_id':    r.get('link_id', '').replace('t3_', ''),
            'created_dt': pd.Timestamp(r['created_utc'], unit='s', tz='UTC'),
        })

comments = pd.DataFrame(comment_rows)
print(f'Comments loaded: {len(comments):,} from {comments["author"].nunique():,} unique authors')
print(f'Date range: {comments["created_dt"].min().date()} → {comments["created_dt"].max().date()}')


Loading comments (this may take ~30s)...


Comments loaded: 500,688 from 80,466 unique authors
Date range: 2023-08-01 → 2025-07-30


In [8]:
# All anchor post IDs across both cycles
all_anchor_ids = anchor_ids_by_cycle[1] | anchor_ids_by_cycle[2]

# Comments on anchor posts
anchor_comments = comments[comments['post_id'].isin(all_anchor_ids)].copy()
print(f'Comments on anchor posts: {len(anchor_comments):,}')

# Tag which cycle each anchor comment belongs to
def comment_cycle(post_id):
    if post_id in anchor_ids_by_cycle[1]: return 1
    if post_id in anchor_ids_by_cycle[2]: return 2
    return None

anchor_comments['cycle'] = anchor_comments['post_id'].apply(comment_cycle)
print(anchor_comments['cycle'].value_counts().sort_index())

Comments on anchor posts: 7,761
cycle
1    3074
2    4687
Name: count, dtype: int64


In [9]:
# Exposed users per cycle: commenters on anchor posts, excluding anchor post authors
# exposure_intensity = max n_dims across all anchor threads the user commented on
exposed_records = []

for cycle in [1, 2]:
    cycle_comments = anchor_comments[anchor_comments['cycle'] == cycle]
    excluded = anchor_authors_by_cycle[cycle]
    eligible = cycle_comments[~cycle_comments['author'].isin(excluded)]

    # For each author, find the max n_dims of anchor posts they commented on
    eligible = eligible.copy()
    eligible['post_ndims'] = eligible['post_id'].map(post_ndims).fillna(1).astype(int)
    author_intensity = eligible.groupby('author')['post_ndims'].max()

    for author, intensity in author_intensity.items():
        exposed_records.append({
            'author': author,
            'exposed': True,
            'cycle': cycle,
            'exposure_intensity': int(intensity)
        })
    print(f'Cycle {cycle} — exposed users: {len(author_intensity):,} '
          f'(excluded {len(set(cycle_comments["author"]) & excluded):,} anchor authors)')
    print(f'  Intensity breakdown: {author_intensity.value_counts().sort_index().to_dict()}')

exposed_df = pd.DataFrame(exposed_records)
print(f'\nTotal exposed: {len(exposed_df):,}')


Cycle 1 — exposed users: 1,189 (excluded 193 anchor authors)
  Intensity breakdown: {1: 268, 2: 156, 3: 765}
Cycle 2 — exposed users: 1,682 (excluded 277 anchor authors)
  Intensity breakdown: {1: 377, 2: 287, 3: 1018}

Total exposed: 2,871


## 5) Identify unexposed users — same-week active, no anchor thread engagement

# Paper §4.3: unexposed = active in r/gradadmissions during the SAME WEEK as an anchor event
# (posted at least one comment or post that week) and did not engage with any anchor thread

In [10]:
import pandas as pd

# Build a set of (author, iso_week) for all posts and comments
# iso_week = 'YYYY-WNN' string — identifies the calendar week
def iso_week(dt):
    """Return 'YYYY-WNN' string for a UTC-aware Timestamp."""
    return dt.strftime('%G-W%V')

posts['iso_week']    = posts['created_dt'].apply(iso_week)
comments['iso_week'] = comments['created_dt'].apply(iso_week)

# Weeks in which each anchor post falls
anchor_posts['iso_week'] = anchor_posts['created_dt'].apply(iso_week)
anchor_weeks_by_cycle = {
    cycle: set(anchor_posts[anchor_posts['cycle'] == cycle]['iso_week'])
    for cycle in [1, 2]
}
print('Anchor weeks — cycle 1:', len(anchor_weeks_by_cycle[1]),
      '| cycle 2:', len(anchor_weeks_by_cycle[2]))

# Users active (posted or commented) in any anchor week, per cycle
unexposed_records = []

for cycle in [1, 2]:
    anchor_weeks = anchor_weeks_by_cycle[cycle]
    exposed_this_cycle = set(exposed_df[exposed_df['cycle'] == cycle]['author'])

    active_post_authors    = set(posts[posts['iso_week'].isin(anchor_weeks)]['author'])
    active_comment_authors = set(comments[comments['iso_week'].isin(anchor_weeks)]['author'])
    active_this_week = active_post_authors | active_comment_authors

    unexposed_authors = active_this_week - exposed_this_cycle

    for author in unexposed_authors:
        unexposed_records.append({
            'author': author,
            'exposed': False,
            'cycle': cycle,
            'exposure_intensity': 0
        })

    print(f'Cycle {cycle} — same-week active: {len(active_this_week):,} | '
          f'exposed: {len(exposed_this_cycle):,} | unexposed: {len(unexposed_authors):,}')

unexposed_df = pd.DataFrame(unexposed_records)
print(f'\nTotal unexposed: {len(unexposed_df):,}')


Anchor weeks — cycle 1: 14 | cycle 2: 14
Cycle 1 — same-week active: 10,326 | exposed: 1,189 | unexposed: 9,226
Cycle 2 — same-week active: 12,915 | exposed: 1,682 | unexposed: 11,295

Total unexposed: 20,521


## 6) Combine and save exposure labels

In [11]:
exposure_df = pd.concat([exposed_df, unexposed_df], ignore_index=True)

# Unexposed users get exposure_intensity = 0
exposure_df['exposure_intensity'] = exposure_df['exposure_intensity'].fillna(0).astype(int)

# A user could appear in both cycles — that's fine, keep both rows
print(f'Total exposure records: {len(exposure_df):,}')
print(f'Unique users: {exposure_df["author"].nunique():,}')
print('\nExposed vs unexposed by cycle:')
print(exposure_df.groupby(['cycle', 'exposed']).size().unstack(fill_value=0))
print('\nExposure intensity distribution (exposed users only):')
exposed_only = exposure_df[exposure_df['exposed']]
print(exposed_only.groupby(['cycle','exposure_intensity'])['author'].count().unstack(fill_value=0))

exposure_df.to_parquet(DATA_DIR / 'exposure_labels_v2.parquet', index=False)
print('\nSaved exposure_labels_v2.parquet')


Total exposure records: 23,392
Unique users: 22,518

Exposed vs unexposed by cycle:
exposed  False  True 
cycle                
1         9226   1189
2        11295   1682

Exposure intensity distribution (exposed users only):
exposure_intensity    1    2     3
cycle                             
1                   268  156   765
2                   377  287  1018

Saved exposure_labels_v2.parquet


## 7) Quick sanity checks

In [12]:
# Users appearing in both cycles
both_cycles = exposure_df.groupby('author')['cycle'].nunique()
print(f'Users active in both cycles: {(both_cycles == 2).sum():,}')

# Users exposed in both cycles
exposed_both = exposure_df[exposure_df['exposed']].groupby('author')['cycle'].nunique()
print(f'Users exposed in both cycles: {(exposed_both == 2).sum():,}')

# Exposure rate per cycle
for cycle in [1, 2]:
    sub = exposure_df[exposure_df['cycle'] == cycle]
    rate = sub['exposed'].mean()
    print(f'Cycle {cycle} exposure rate: {rate:.2%} ({sub["exposed"].sum():,} exposed / {len(sub):,} active)')

Users active in both cycles: 874
Users exposed in both cycles: 67
Cycle 1 exposure rate: 11.42% (1,189 exposed / 10,415 active)
Cycle 2 exposure rate: 12.96% (1,682 exposed / 12,977 active)
